## Setup and MLflow Initialization
We connect to DagsHub and set the experiment to **RandomForest_Training**. Every run in this notebook is grouped under that experiment so results are clearly separated from other architectures on DagsHub.

In [4]:
!pip install mlflow dagshub -q

In [5]:
import os

# Data path for this Kaggle environment
DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'
print(os.listdir(DATA_PATH))

['sample_submission.csv', 'test_identity.csv', 'train_identity.csv', 'test_transaction.csv', 'train_transaction.csv']


In [6]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Connect to DagsHub
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML-Assignment-2')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow")
mlflow.set_experiment("RandomForest_Training")

print("DagsHub connection established.")
print("Experiment: RandomForest_Training")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=50b256d2-27d2-405e-a923-e16225379756&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e5de26686f079fced0211423adbf90185fe52bf5476550bd93462ee3c0b20942




Output()

Accessing as GigiSichinava

Initialized MLflow to track repo "GigiSichinava/ML-Assignment-2"

Repository GigiSichinava/ML-Assignment-2 initialized!

2026/05/03 17:52:00 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Training' does not exist. Creating a new experiment.


DagsHub connection established.
Experiment: RandomForest_Training


## Data Loading
We load both the transaction and identity tables then merge them on TransactionID with a left join. This keeps all transactions even those with no identity record.

In [7]:
# Load and merge the two tables on TransactionID
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')


# Left join keeps all transactions even those without an identity record
train_df = train_transaction.merge(train_identity, on='TransactionID', how='left')
test_df  = test_transaction.merge(test_identity,  on='TransactionID', how='left')

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Fraud rate:  {train_df['isFraud'].mean():.4f}  ({train_df['isFraud'].sum()} fraud cases)")


Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  0.0350  (20663 fraud cases)


## Data Cleaning
Three cleaning steps are applied and logged as a single MLflow run. First we drop columns where more than 50% of values are missing. Then numeric NaNs are filled with the column median — robust to the heavy skew in financial data. Categorical NaNs are filled with the column mode.

In [8]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):
    mlflow.log_param("stage", "Cleaning")

    # Separate target and drop the ID column
    y = train_df['isFraud'].copy()
    train_df.drop(columns=['isFraud', 'TransactionID'], inplace=True, errors='ignore')
    test_df.drop(columns=['TransactionID'], inplace=True, errors='ignore')

    # Step 1 - Drop columns where more than 50% of values are missing
    # These columns carry too little information to be useful after imputation
    missing_ratio = train_df.isnull().mean()
    high_missing  = missing_ratio[missing_ratio > 0.5].index.tolist()
    train_df.drop(columns=high_missing, inplace=True)
    test_df.drop(columns=[c for c in high_missing if c in test_df.columns], inplace=True)
    print(f"Dropped {len(high_missing)} high-missing columns")

    # Identify numeric and categorical columns after dropping
    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

    # Step 2 - Fill numeric NaN with median
    # Median is more robust than mean because transaction amounts are heavily skewed
    for col in num_cols:
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        test_df[col]  = test_df[col].fillna(median_val)

    # Step 3 - Fill categorical NaN with mode
    # The most frequent category is the safest default for unknown values
    for col in cat_cols:
        mode_val = train_df[col].mode()[0]
        train_df[col] = train_df[col].fillna(mode_val)
        test_df[col]  = test_df[col].fillna(mode_val)

    mlflow.log_param("dropped_columns",   len(high_missing))
    mlflow.log_param("numeric_features",  len(num_cols))
    mlflow.log_param("categorical_features", len(cat_cols))
    print(f"After cleaning: Train={train_df.shape}, Test={test_df.shape}")
    print("Cleaning run logged to MLflow.")


Dropped 214 high-missing columns
After cleaning: Train=(590540, 218), Test=(506691, 256)
Cleaning run logged to MLflow.
🏃 View run RandomForest_Cleaning at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1/runs/c737e9de09e14db29c90ffbd42cad65b
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1


## Feature Engineering
We create five new features that capture fraud signals not visible in the raw columns. Log-transformed amount corrects for skew. The decimal part captures suspicious cent patterns. Hour and day features expose temporal fraud behaviour. All categorical columns are label-encoded since tree models handle integer inputs well.

In [9]:
# Feature 1 - Log transform of transaction amount
# Transaction amounts are right skewed so log1p brings the distribution closer to normal
train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
test_df['TransactionAmt_log']  = np.log1p(test_df['TransactionAmt'])

# Feature 2 - Decimal part of transaction amount
# Fraudsters often use amounts like 99.00 or 0.01 — the cents part captures this
train_df['TransactionAmt_decimal'] = train_df['TransactionAmt'] - train_df['TransactionAmt'].astype(int)
test_df['TransactionAmt_decimal']  = test_df['TransactionAmt'] - test_df['TransactionAmt'].astype(int)

# Feature 3 - Hour of day extracted from TransactionDT timestamp
# Fraud tends to happen at unusual hours like late night
train_df['Transaction_hour'] = (train_df['TransactionDT'] // 3600) % 24
test_df['Transaction_hour']  = (test_df['TransactionDT']  // 3600) % 24

# Feature 4 - Day of week extracted from TransactionDT
# Fraud patterns often differ by day of week
train_df['Transaction_day'] = (train_df['TransactionDT'] // (3600 * 24)) % 7
test_df['Transaction_day']  = (test_df['TransactionDT']  // (3600 * 24)) % 7

# Feature 5 - Label encode all categorical columns
# Tree models handle integer encoded categories well without one-hot expansion
le = LabelEncoder()
for col in cat_cols:
    if col in train_df.columns:
        combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
        le.fit(combined)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))

print(f"Feature engineering complete. Final shape: {train_df.shape}")


/tmp/ipykernel_57/703187690.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
/tmp/ipykernel_57/703187690.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df['TransactionAmt_log']  = np.log1p(test_df['TransactionAmt'])
/tmp/ipykernel_57/703187690.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(a

Feature engineering complete. Final shape: (590540, 222)


## Feature Selection
Three sequential methods are applied and logged to MLflow. Variance threshold removes near-constant columns. Correlation filter removes redundant pairs above 0.95. Random Forest importance then selects the top 50 most predictive features.

In [10]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="RandomForest_Feature_Selection"):
    mlflow.log_param("stage", "Feature_Selection")

    X         = train_df.copy()
    X_test_fs = test_df.copy()

    # Method 1 - Variance threshold removes near-constant columns
    # A feature with almost no variance cannot help the model distinguish fraud
    vt           = VarianceThreshold(threshold=0.01)
    vt.fit(X)
    low_var_cols = X.columns[~vt.get_support()].tolist()
    X            = X.drop(columns=low_var_cols)
    X_test_fs    = X_test_fs.drop(columns=[c for c in low_var_cols if c in X_test_fs.columns])
    print(f"After variance threshold: {X.shape[1]} features")

    # Method 2 - Correlation filter removes redundant feature pairs above 0.95
    # Highly correlated features provide duplicate information and can slow training
    corr_matrix    = X.corr().abs()
    upper          = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
    X              = X.drop(columns=high_corr_cols)
    X_test_fs      = X_test_fs.drop(columns=[c for c in high_corr_cols if c in X_test_fs.columns])
    print(f"After correlation filter: {X.shape[1]} features")

    # Method 3 - Random Forest importance keeps the top 50 most predictive features
    rf_selector  = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    rf_selector.fit(X, y)
    importances  = pd.Series(rf_selector.feature_importances_, index=X.columns)
    top_features = importances.nlargest(50).index.tolist()
    X_selected      = X[top_features]
    X_test_selected = X_test_fs[top_features]
    print(f"Top 50 features selected by Random Forest importance")

    mlflow.log_param("features_after_variance", X.shape[1])
    mlflow.log_param("features_after_corr",     X.shape[1] - len(high_corr_cols))
    mlflow.log_param("final_feature_count",      len(top_features))

    # Stratified split preserves the fraud ratio in both train and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_selected, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")
    print("Feature selection run logged to MLflow.")


After variance threshold: 199 features
After correlation filter: 144 features
Top 50 features selected by Random Forest importance
Train: (472432, 50), Val: (118108, 50)
Feature selection run logged to MLflow.
🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1/runs/94ab402d879d4cf194d83ff6e675271a
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1


## Model Training
We use the same `train_and_log` helper pattern from Assignment 1 to train and log each configuration. Multiple hyperparameter values are tested to observe underfitting, overfitting, and the optimal balance between them. All runs are logged to the **RandomForest_Training** experiment on DagsHub.

In [11]:
# Helper function — same pattern as Assignment 1
# Trains the model, computes metrics, logs everything to MLflow and saves to registry
def train_and_log(model, model_name):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        train_preds = model.predict_proba(X_train)[:, 1]
        val_preds   = model.predict_proba(X_val)[:, 1]

        train_auc = roc_auc_score(y_train, train_preds)
        val_auc   = roc_auc_score(y_val,   val_preds)
        gap       = train_auc - val_auc

        mlflow.log_param("model_type",  model_name)
        mlflow.log_metric("train_auc",  train_auc)
        mlflow.log_metric("val_auc",    val_auc)
        mlflow.log_metric("overfit_gap", gap)

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name
        )

        print(f"Model logged: {model_name}")
        print(f"   Train AUC:   {train_auc:.4f}")
        print(f"   Val AUC:     {val_auc:.4f}")
        print(f"   Overfit gap: {gap:.4f}")
        print("-" * 30)

        return model


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# n_estimators sweep
rf_options = [50, 100, 200, 300]
print("=== n_estimators Sweep ===")
for n in rf_options:
    label = f"RF_n_estimators_{n}"
    model = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=label):
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        model.fit(X_train, y_train)
        train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
        val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
        gap       = train_auc - val_auc
        mlflow.log_param("model_type",  label)
        mlflow.log_param("n_estimators", n)
        mlflow.log_metric("train_auc",   train_auc)
        mlflow.log_metric("val_auc",     val_auc)
        mlflow.log_metric("cv_auc_mean", cv_scores.mean())
        mlflow.log_metric("cv_auc_std",  cv_scores.std())
        mlflow.log_metric("overfit_gap", gap)
        results.append({"label": label, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"{label}: Train={train_auc:.4f} Val={val_auc:.4f} Gap={gap:.4f}")
    print("-" * 30)

# max_depth sweep — underfitting to overfitting
rf_depths = [3, 5, 10, 15, 20, None]
print("\n=== max_depth Sweep ===")
for depth in rf_depths:
    label = f"RF_depth_{'None' if depth is None else depth}"
    model = RandomForestClassifier(n_estimators=200, max_depth=depth, random_state=42, n_jobs=-1)
    with mlflow.start_run(run_name=label):
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        model.fit(X_train, y_train)
        train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
        val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
        gap       = train_auc - val_auc
        mlflow.log_param("model_type", label)
        mlflow.log_param("max_depth",  str(depth))
        mlflow.log_metric("train_auc",   train_auc)
        mlflow.log_metric("val_auc",     val_auc)
        mlflow.log_metric("cv_auc_mean", cv_scores.mean())
        mlflow.log_metric("cv_auc_std",  cv_scores.std())
        mlflow.log_metric("overfit_gap", gap)
        results.append({"label": label, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"{label}: Train={train_auc:.4f} Val={val_auc:.4f} Gap={gap:.4f}")
        if depth is None:
            print("   Note: Unlimited depth — overfitting expected.")
        if depth is not None and depth <= 5:
            print("   Note: Shallow depth — likely underfitting.")
    print("-" * 30)

print("\nRandom Forest training complete.")


=== n_estimators Sweep ===
RF_n_estimators_50: Train=1.0000 Val=0.9130 Gap=0.0870
🏃 View run RF_n_estimators_50 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1/runs/98f6fd671b2840b289b8607469ff8618
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1
------------------------------
RF_n_estimators_100: Train=1.0000 Val=0.9186 Gap=0.0814
🏃 View run RF_n_estimators_100 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1/runs/9522c44e16944a38a72bfb0b793a7b40
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1
------------------------------
RF_n_estimators_200: Train=1.0000 Val=0.9210 Gap=0.0790
🏃 View run RF_n_estimators_200 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1/runs/fcc3a32c413c42748a5137bc233e0b16
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1
----------------------

KeyboardInterrupt: 

## Results Visualization
Train vs Validation AUC and the overfit gap are plotted for every configuration. Red bars in the gap chart indicate configurations where the model is overfitting (gap above 0.05).

In [13]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

labels     = [r["label"] for r in results]
train_aucs = [r["train_auc"] for r in results]
val_aucs   = [r["val_auc"]   for r in results]
gaps       = [r["gap"]       for r in results]
x = list(range(len(labels)))

# Plot 1 — Train vs Validation AUC per configuration
axes[0].bar([i - 0.2 for i in x], train_aucs, 0.4, label="Train AUC", color="steelblue", alpha=0.8)
axes[0].bar([i + 0.2 for i in x], val_aucs,   0.4, label="Val AUC",   color="coral",     alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([l.replace("RF_", "") for l in labels], rotation=45, ha="right", fontsize=8)
axes[0].set_ylim(0.5, 1.02)
axes[0].set_ylabel("ROC-AUC")
axes[0].set_title("RandomForest — Train vs Validation AUC")
axes[0].legend()

# Plot 2 — Overfit gap per configuration (red = overfitting, green = ok)
colors = ["red" if g > 0.05 else "green" for g in gaps]
axes[1].bar(x, gaps, color=colors, alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([l.replace("RF_", "") for l in labels], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Train AUC - Val AUC")
axes[1].set_title("RandomForest — Overfit Gap (red = gap above 0.05)")
axes[1].axhline(y=0.05, color="red", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig("rf_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved.")


Plot saved.


## Best Pipeline Registration
The best RandomForest configuration is wrapped in a sklearn Pipeline together with a preprocessing function. This pipeline can run directly on raw unprocessed data because it applies all cleaning and feature engineering steps internally. The pipeline is saved to the DagsHub Model Registry.

In [14]:
import numpy as np
from sklearn.preprocessing import LabelEncoder


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# This function applies the exact same steps as the cleaning and feature
# engineering cells above, so the pipeline can run on raw unprocessed data
def preprocess_raw(df):
    df = df.copy()

    # Drop high-missing columns using the same 50% threshold
    missing_ratio = df.isnull().mean()
    df = df.drop(columns=missing_ratio[missing_ratio > 0.5].index.tolist(), errors='ignore')

    num_c = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_c = df.select_dtypes(include=['object']).columns.tolist()

    # Fill missing values the same way as in the Cleaning section
    for col in num_c:
        df[col] = df[col].fillna(df[col].median())
    for col in cat_c:
        df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')

    # Apply the same engineered features
    df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
    df['Transaction_hour']       = (df['TransactionDT'] // 3600) % 24
    df['Transaction_day']        = (df['TransactionDT'] // (3600 * 24)) % 7

    # Label encode categoricals
    le = LabelEncoder()
    for col in cat_c:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    # Keep only the top features identified in the Feature Selection section
    keep = [c for c in top_features if c in df.columns]
    return df[keep]

from sklearn.ensemble import RandomForestClassifier
best_rf = RandomForestClassifier(
    n_estimators=300, max_depth=20, max_features='sqrt',
    min_samples_leaf=4, random_state=42, n_jobs=-1
)

# Wrap preprocessing + model into a Pipeline
# This pipeline runs directly on raw unprocessed data
randomforest_pipeline = Pipeline([
    ('preprocessor', FunctionTransformer(preprocess_raw)),
    ('classifier',   best_rf)
])

# Fit the pipeline on the original raw training data
train_raw = train_transaction.merge(train_identity, on='TransactionID', how='left')
y_raw     = train_raw['isFraud'].copy()
train_raw = train_raw.drop(columns=['isFraud', 'TransactionID'], errors='ignore')

randomforest_pipeline.fit(train_raw, y_raw)
print("RandomForest Pipeline trained on raw data successfully.")

# Register the pipeline to DagsHub Model Registry
with mlflow.start_run(run_name="RandomForest_Pipeline_Registration"):
    mlflow.log_param("model_type", "RandomForest_Pipeline")
    mlflow.sklearn.log_model(
        sk_model=randomforest_pipeline,
        artifact_path="model",
        registered_model_name="RandomForest_Pipeline"
    )
    print("RandomForest Pipeline registered in DagsHub Model Registry.")


/tmp/ipykernel_57/4010930922.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
/tmp/ipykernel_57/4010930922.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
/tmp/ipykernel_57/4010930922.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u

RandomForest Pipeline trained on raw data successfully.


2026/05/03 18:54:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 18:55:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'RandomForest_Pipeline'.
2026/05/03 18:55:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_Pipeline, version 1
Created version '1' of model 'RandomForest_Pipeline'.


RandomForest Pipeline registered in DagsHub Model Registry.
🏃 View run RandomForest_Pipeline_Registration at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1/runs/113a269cfcae47aa94e1a00440ca9f05
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/1
